In [1]:
import pandas as pd
import torch
from sklearn.model_selection import train_test_split

In [2]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM (GB):", torch.cuda.get_device_properties(0).total_memory / 1e9)

CUDA available: True
GPU: NVIDIA A100-SXM4-80GB
VRAM (GB): 85.167243264


In [3]:
DEV_PROCESSED  = "/content/development_processed.csv"
EVAL_PROCESSED = "/content/evaluation_processed.csv"

df_dev  = pd.read_csv(DEV_PROCESSED)
df_eval = pd.read_csv(EVAL_PROCESSED)
MAX_LEN = 512
print(df_dev.shape, df_eval.shape)

(79997, 12) (20000, 11)


In [4]:
def build_transformer_text(df):
    return (
        df["title"].astype(str) +
        "\n\n" +
        df["article"].astype(str)
    )

df_dev["tr_text"]  = build_transformer_text(df_dev)
df_eval["tr_text"] = build_transformer_text(df_eval)


In [5]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    df_dev,
    test_size=0.15,
    stratify=df_dev["label"],
    random_state=42
)

print("Train:", train_df.shape)
print("Val:", val_df.shape)


Train: (67997, 13)
Val: (12000, 13)


In [6]:
class NewsDataset(torch.utils.data.Dataset):
    def __init__(self, df, tokenizer, max_length=MAX_LEN):
        self.texts  = df["tr_text"].tolist()
        self.labels = df["label"].values if "label" in df else None
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        item = {k: v.squeeze(0) for k, v in enc.items()}

        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)

        return item

In [7]:
from transformers import RobertaTokenizer

MODEL_NAME = "roberta-large"
tokenizer = RobertaTokenizer.from_pretrained(MODEL_NAME)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

In [8]:
train_ds = NewsDataset(train_df, tokenizer)
val_ds   = NewsDataset(val_df, tokenizer)
eval_ds  = NewsDataset(df_eval, tokenizer)

In [9]:
from transformers import RobertaForSequenceClassification

model = RobertaForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=7
).cuda()

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-large and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [10]:
from transformers import TrainingArguments

args = TrainingArguments(
    output_dir="./roberta_out",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    fp16=True,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",

    logging_steps=100,
    report_to="none",
    save_total_limit=2
)


In [11]:
from transformers import Trainer, TrainingArguments


In [12]:
from sklearn.metrics import f1_score
import numpy as np

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "macro_f1": f1_score(labels, preds, average="macro")
    }


In [13]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics
)

In [14]:
trainer.train()

Epoch,Training Loss,Validation Loss,Macro F1
1,0.729500,0.663570,0.727810
2,0.596800,0.634881,0.743329
3,0.468200,0.658592,0.755512


TrainOutput(global_step=25500, training_loss=0.6307975779514686, metrics={'train_runtime': 3015.8051, 'train_samples_per_second': 67.641, 'train_steps_per_second': 8.455, 'total_flos': 1.9010882237094605e+17, 'train_loss': 0.6307975779514686, 'epoch': 3.0})

In [15]:
load_best_model_at_end=True
metric_for_best_model="macro_f1"

In [16]:
full_train_df = df_dev.copy()

full_train_ds = NewsDataset(
    full_train_df,
    tokenizer,
    max_length=MAX_LEN
)

In [17]:
from transformers import TrainingArguments

args_full = TrainingArguments(
    output_dir="./roberta_full_out",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=2,
    num_train_epochs=3,
    fp16=True,
    eval_strategy="no",
    save_strategy="no",
    logging_steps=100,
    report_to="none"
)


In [18]:
trainer_full = Trainer(
    model=trainer.model,
    args=args_full,
    train_dataset=full_train_ds
)


In [19]:
trainer_full.train()

Step,Training Loss
100,0.493500
200,0.487700
300,0.494500
400,0.498200
500,0.509300
600,0.512900
700,0.517600
800,0.489400
900,0.508300
1000,0.511700


TrainOutput(global_step=7500, training_loss=0.4184545605977376, metrics={'train_runtime': 2717.3778, 'train_samples_per_second': 88.317, 'train_steps_per_second': 2.76, 'total_flos': 2.2365891823475405e+17, 'train_loss': 0.4184545605977376, 'epoch': 3.0})

In [20]:
preds = trainer_full.predict(eval_ds)
logits = preds.predictions
final_pred = logits.argmax(axis=1)

In [21]:
pred_out = trainer.predict(eval_ds)
logits = pred_out.predictions

np.save(f"logits_roberta_processed_noseed_MAXLEN_{MAX_LEN}.npy", logits)

preds = logits.argmax(axis=1)

submission = pd.DataFrame({
	"Id": df_eval["Id"].astype(int),
	"Predicted": preds.astype(int)
})

submission.to_csv(f"submission_roberta_processed_noseed_MAXLEN_{MAX_LEN}.csv", index=False)

print(f"Saved submission_noseed_MAX_LEN{MAX_LEN}.csv")
print(f"Saved logits_noseed_MAX_LEN{MAX_LEN}.npy")

Saved submission_noseed_MAX_LEN512.csv
Saved logits_noseed_MAX_LEN512.npy
